In [ ]:

import os, cv2, dlib, math
import numpy as np
from tqdm.notebook import tqdm
from scipy.stats import skew
from skimage.feature import local_binary_pattern
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve
import matplotlib.pyplot as plt

# ----- Load Image & Landmarks -----
detector = dlib.get_frontal_face_detector()

predictor = dlib.shape_predictor('shape_predictor_68_face_landmarks.dat')

def load_image(path):
    img = cv2.imread(path)
    if img is None: return None
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def get_landmarks(img, predictor):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    rects = detector(gray, 1)
    if len(rects)==0: return None
    rect = rects[0]
    shape = predictor(gray, rect)
    coords = np.array([[p.x, p.y] for p in shape.parts()])
    return coords

# ----- Face Alignment & ROI Extraction -----
def align_crop(img, landmarks):
    left_eye = np.mean(landmarks[36:42], axis=0)
    right_eye = np.mean(landmarks[42:48], axis=0)
    dy = right_eye[1]-left_eye[1]
    dx = right_eye[0]-left_eye[0]
    angle = math.degrees(math.atan2(dy,dx))
    eyes_center = ((left_eye[0]+right_eye[0])//2, (left_eye[1]+right_eye[1])//2)
    M = cv2.getRotationMatrix2D(tuple(eyes_center), angle, 1.0)
    rotated = cv2.warpAffine(img, M, (img.shape[1], img.shape[0]), flags=cv2.INTER_CUBIC)
    l = int(eyes_center[0]-75); t = int(eyes_center[1]-60)
    w,h = 150,130
    h_img,w_img = rotated.shape[:2]
    l=max(0,l); t=max(0,t)
    if l+w> w_img: l = w_img-w
    if t+h> h_img: t = h_img-h
    crop = rotated[t:t+h, l:l+w]
    if crop.shape[0]!=h or crop.shape[1]!=w:
        crop = cv2.resize(crop, (w,h))
    return crop

def extract_rois(face):
    fh, fw = face.shape[:2]
    lx, ly, lw, lh = 20, 20, 52, 52
    rx, ry = 78, 20
    mx, my, mw, mh = 46, 80, 56, 62
    left = cv2.resize(face[ly:ly+lh, lx:lx+lw], (52,52))
    right = cv2.resize(face[ry:ry+lh, rx:rx+lw], (52,52))
    mouth = cv2.resize(face[my:my+mh, mx:mx+mw], (56,62))
    return left, right, mouth

# ----- Feature Extraction -----
def color_moments_patch(img, blocks_x, blocks_y):
    HSV = cv2.cvtColor(img, cv2.COLOR_RGB2HSV).astype(np.float32)
    h,w,_ = HSV.shape
    bx = w//blocks_x; by = h//blocks_y
    feats = []
    for byi in range(blocks_y):
        for bxi in range(blocks_x):
            x0 = bxi*bx; y0 = byi*by
            patch = HSV[y0:y0+by, x0:x0+bx]
            for ch in range(3):
                v = patch[:,:,ch].ravel()
                if v.size==0: feats += [0.0,0.0,0.0]
                else: feats += [np.mean(v), np.std(v), skew(v)]
    return np.array(feats)

def color_features_face(face):
    return color_moments_patch(face, 3, 3)

def color_features_roi(roi):
    return color_moments_patch(roi, 5, 5)

# ----- Gabor -----
def make_gabor_kernels(scales, orientations, ksize=31):
    kernels = []
    for s in range(scales):
        for o in range(orientations):
            theta = o * np.pi / orientations
            sigma = 2.0 + s*0.5
            lamda = 3.0 + s*1.0
            gamma = 0.5
            kern = cv2.getGaborKernel((ksize,ksize), sigma, theta, lamda, gamma, 0, ktype=cv2.CV_32F)
            kernels.append(kern)
    return kernels

def gabor_features(face):
    gray = cv2.cvtColor(face, cv2.COLOR_RGB2GRAY).astype(np.float32)
    kernels = make_gabor_kernels(5,8)
    feats=[]
    for k in kernels:
        f = cv2.filter2D(gray, cv2.CV_32F, k)
        v = f.ravel()
        feats += [np.mean(v), np.std(v), skew(v)]
    return np.array(feats)

# ----- Other descriptors -----
def gist_descriptor(face):
    gray = cv2.cvtColor(face, cv2.COLOR_RGB2GRAY).astype(np.float32)
    filt = make_gabor_kernels(4,8)
    h,w = gray.shape
    desc=[]
    for k in filt:
        resp = cv2.filter2D(gray, cv2.CV_32F, k)
        block_h, block_w = h//4, w//4
        for r in range(4):
            for c in range(4):
                y0 = r*block_h; x0 = c*block_w
                desc.append(np.mean(resp[y0:y0+block_h, x0:x0+block_w]))
    return np.array(desc)

def eoh_descriptor(face, bins=37):
    gray = cv2.cvtColor(face, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, 100,200)
    sobelx = cv2.Sobel(gray, cv2.CV_32F,1,0,ksize=3)
    sobely = cv2.Sobel(gray, cv2.CV_32F,0,1,ksize=3)
    mag = np.hypot(sobelx, sobely)
    ang = np.degrees(np.arctan2(sobely, sobelx)) % 180
    mask = edges>0
    if np.sum(mask)==0: return np.zeros(bins)
    hist,_ = np.histogram(ang[mask], bins=bins, range=(0,180))
    hist = hist.astype(np.float32); hist /= (np.sum(hist)+1e-8)
    return hist

def lbp_descriptor(face):
    gray = cv2.cvtColor(face, cv2.COLOR_RGB2GRAY)
    lbp = local_binary_pattern(gray, 8, 1, method='uniform')
    hist,_ = np.histogram(lbp.ravel(), bins=59, range=(0,59))
    hist = hist.astype('float') / (hist.sum()+1e-8)
    return hist

# ----- Feature Vector -----
def extract_feature_vector(img, predictor):
    lm = get_landmarks(img, predictor)
    if lm is None: return None
    face = align_crop(img, lm)
    left, right, mouth = extract_rois(face)
    face_vec = np.concatenate([color_features_face(face), gabor_features(face), gist_descriptor(face), eoh_descriptor(face), lbp_descriptor(face)])
    roi_vec = np.concatenate([color_features_roi(left), color_features_roi(right), color_features_roi(mouth)])
    return np.concatenate([face_vec, roi_vec])

# ----- Build Dataset -----
def build_dataset(data_dir):
    X=[]; y=[]
    for lbl,cls in enumerate(['no_makeup','makeup']):
        folder = os.path.join(data_dir, cls)
        files = [os.path.join(folder,f) for f in os.listdir(folder) if f.lower().endswith(('.jpg','.png'))]
        for f in tqdm(files, desc=cls):
            img = load_image(f)
            if img is None: continue
            fv = extract_feature_vector(img, predictor)
            if fv is None: continue
            X.append(fv); y.append(lbl)
    return np.array(X), np.array(y)

# ----- Train & Evaluate -----
def train_evaluate(X,y):
    scaler = StandardScaler(); Xs = scaler.fit_transform(X)
    param_grid = {'C':[1,8,32,128], 'gamma':[1.2e-4,2e-3,0.002,0.00048]}
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    grid = GridSearchCV(SVC(kernel='rbf', probability=True), param_grid, cv=cv, scoring='accuracy', n_jobs=1)
    grid.fit(Xs,y)
    print("Best params:", grid.best_params_, "Best score:", grid.best_score_)
    accs=[]; aucs=[]; probas=np.zeros((X.shape[0],))
    for train_idx,test_idx in cv.split(Xs,y):
        clf = SVC(kernel='rbf', C=grid.best_params_['C'], gamma=grid.best_params_['gamma'], probability=True)
        clf.fit(Xs[train_idx], y[train_idx])
        preds = clf.predict(Xs[test_idx]); accs.append(accuracy_score(y[test_idx], preds))
        probs = clf.predict_proba(Xs[test_idx])[:,1]
        try: a=roc_auc_score(y[test_idx], probs)
        except: a=0.5
        aucs.append(a); probas[test_idx]=probs
    print("CV Acc Avg:", np.mean(accs), "AUC Avg:", np.mean(aucs))
    fpr,tpr,_=roc_curve(y,probas); plt.plot(fpr,tpr); plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title('ROC'); plt.show()

# ----- Main Execution -----
data_dir = 'dataset'  # replace with your dataset path
X,y = build_dataset(data_dir)
print("Features shape:", X.shape, "Labels:", np.unique(y, return_counts=True))
train_evaluate(X,y)
